**1. Load all results: for each k (number of recommended books), adjust the relevant settings, run 02_experiments.ipynb, and move results to ../results/@k/**

In [ ]:
import json

targets = ["@5", "@10", "@15"]
result_ground_truth = {}
result_baseline_bm25 = {}
result_baseline_librarian = {}
result_sbs = {}

# init
for target in targets:
    result_ground_truth[target] = []
    result_baseline_bm25[target] = []
    result_baseline_librarian[target] = []
    result_sbs[target] = []

for target in targets:
    for i in range(1, 101):
        filename = f"../results/{target}/ground_truth/{i:03}.json"
        with open(filename, "r", encoding="utf-8") as f:
            result_ground_truth[target].append(json.load(f))
        filename = f"../results/{target}/baseline_bm25/{i:03}.json"
        with open(filename, "r", encoding="utf-8") as f:
            result_baseline_bm25[target].append(json.load(f))
        filename = f"../results/{target}/baseline_librarian/{i:03}.json"
        with open(filename, "r", encoding="utf-8") as f:
            result_baseline_librarian[target].append(json.load(f))
        filename = f"../results/{target}/smart_book_seeker/{i:03}.json"
        with open(filename, "r", encoding="utf-8") as f:
            result_sbs[target].append(json.load(f))

print(json.dumps(result_ground_truth, ensure_ascii=False , indent=4))
print(json.dumps(result_baseline_bm25, ensure_ascii=False , indent=4))
print(json.dumps(result_baseline_librarian, ensure_ascii=False , indent=4))
print(json.dumps(result_sbs, ensure_ascii=False , indent=4))

**2. Remove books with incorrect quantities**

In [ ]:
for target in targets:
    for i in range(100):
        if len(result_baseline_bm25[target][i]["books"]) > int(target[1:]):
            print(f"{target}/bm25: Requirement {i} exceeds the limit: {len(result_baseline_bm25[target][i]['books'])} books found, {target[1:]} books maximum")
            result_baseline_bm25[target][i]["books"] = result_baseline_bm25[target][i]["books"][:int(target[1:])]
        if len(result_baseline_librarian[target][i]["books"]) > int(target[1:]):
            print(f"{target}/bm25: Requirement {i} exceeds the limit: {len(result_baseline_librarian[target][i]['books'])} books found, {target[1:]} books maximum")
            result_baseline_librarian[target][i]["books"] = result_baseline_librarian[target][i]["books"][:int(target[1:])]
        if len(result_sbs[target][i]["books"]) > int(target[1:]):
            print(f"{target}/bm25: Requirement {i} exceeds the limit: {len(result_sbs[target][i]['books'])} books found, {target[1:]} books maximum")
            result_sbs[target][i]["books"] = result_sbs[target][i]["books"][:int(target[1:])]

**3. Calculate precision**

In [ ]:
precision_baseline_bm25 = {}
precision_baseline_librarian = {}
precision_sbs = {}

# init
for target in targets:
    precision_baseline_bm25[target] = []
    precision_baseline_librarian[target] = []
    precision_sbs[target] = []

for target in targets:
    for i in range(100):
        temp_bm25 = 0
        temp_librarian = 0
        temp_sbs = 0

        for ans_book in result_ground_truth[target][i]["books"]:
            for book in result_baseline_bm25[target][i]["books"]:
                if book["id"] == ans_book["id"]:
                    temp_bm25 += 1 / int(target[1:])
                    break
            for book in result_baseline_librarian[target][i]["books"]:
                if book["id"] == ans_book["id"]:
                    temp_librarian += 1 / int(target[1:])
                    break
            for book in result_sbs[target][i]["books"]:
                if book["id"] == ans_book["id"]:
                    temp_sbs += 1 / int(target[1:])
                    break

        precision_baseline_bm25[target].append(temp_bm25)
        precision_baseline_librarian[target].append(temp_librarian)
        precision_sbs[target].append(temp_sbs)

avg_precision_baseline_bm25 = {}
avg_precision_baseline_librarian = {}
avg_precision_sbs = {}

# init
for target in targets:
    avg_precision_baseline_bm25[target] = 0
    avg_precision_baseline_librarian[target] = 0
    avg_precision_sbs[target] = 0

for target in targets:
    for i in range(100):
        avg_precision_baseline_bm25[target] += precision_baseline_bm25[target][i] / 100
        avg_precision_baseline_librarian[target] += precision_baseline_librarian[target][i] / 100
        avg_precision_sbs[target] += precision_sbs[target][i] / 100

print(avg_precision_baseline_bm25)
print(avg_precision_baseline_librarian)
print(avg_precision_sbs)

**4. Calculate standard deviation of precision**

In [ ]:
import numpy as np

std_bm25 = {}
std_librarian = {}
std_sbs = {}

for target in targets:
    std_bm25[target] = np.std(precision_baseline_bm25[target]).item()
    std_librarian[target] = np.std(precision_baseline_librarian[target]).item()
    std_sbs[target] = np.std(precision_sbs[target]).item()

print(std_bm25)
print(std_librarian)
print(std_sbs)

**5. Find best case**

In [ ]:
max_target = ""
max_gap = 0
max_needs_id = 0
max_turn = 0

for target in targets:
    for i in range(100):
        if len(result_baseline_librarian[target][i]["books"]) != int(target[1:]) or len(result_sbs[target][i]["books"]) != int(target[1:]):
            continue
        if precision_sbs[target][i] - precision_baseline_librarian[target][i] > max_gap and int(result_sbs[target][i]["turn"]) > 2:

            max_needs_id = i
            max_target = target
            max_gap = precision_sbs[target][i] - precision_baseline_librarian[target][i]
            max_turn = int(result_sbs[target][i]["turn"])

print(max_needs_id)
print(max_target)
print(max_gap)
print(max_turn)

print(precision_baseline_librarian[max_target][max_needs_id])
print(precision_sbs[max_target][max_needs_id])

print(result_baseline_librarian[max_target][max_needs_id]["user_book_needs"])


**6. Find worst case**

In [ ]:
max_target = ""
max_gap = 0
max_needs_id = 0

for target in targets:
    for i in range(100):
        if len(result_baseline_librarian[target][i]["books"]) != int(target[1:]) or len(result_sbs[target][i]["books"]) != int(target[1:]):
            continue
        if precision_baseline_librarian[target][i] - precision_sbs[target][i] > max_gap:
            max_needs_id = i
            max_target = target
            max_gap = precision_baseline_librarian[target][i] - precision_sbs[target][i]

print(max_needs_id)
print(max_target)
print(max_gap)

print(precision_baseline_librarian[max_target][max_needs_id])
print(precision_sbs[max_target][max_needs_id])

print(result_baseline_librarian[max_target][max_needs_id]["user_book_needs"])


**7. Draw bot plot for precision@5**

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('default')
data   = [precision_baseline_bm25["@5"], precision_baseline_librarian["@5"], precision_sbs["@5"]]
labels = ['Keyword-based IR', 'Librarian Agent', 'SBS-AARS']

# 顯示完整統計資訊
for i, d in enumerate(data, 1):
    stats = {
        'median': np.median(d).item(),
        'mean': np.mean(d).item(),
        'q1': np.percentile(d, 25).item(),
        'q3': np.percentile(d, 75).item()
    }

    print(stats)

plt.figure(figsize=(6, 4), dpi=300)
box = plt.boxplot(
    data,
    patch_artist=True,
    showfliers=False,
    boxprops=dict(color='black'),
    medianprops=dict(color='black', linewidth=1.5),
    whiskerprops=dict(color='black'),
    capprops=dict(color='black')
)
hatches = ['...', '///', '']
for patch, hatch in zip(box['boxes'], hatches):
    patch.set_facecolor('white')
    patch.set_hatch(hatch)
    patch.set_edgecolor('black')
plt.xticks([1, 2, 3], labels)
plt.ylabel('Precision')
plt.grid(True, axis='y', linestyle='--', alpha=0.7)
# plt.title('Baseline v.s. SBS-AARS')
# legend_elements = [
#     Patch(facecolor='white', edgecolor='black', hatch=hatches[0],
#           label=f'Baseline (Average = {avg1:.2f}%)'),
#     Patch(facecolor='white', edgecolor='black', hatch=hatches[1],
#           label=f'Smart Book Seeker (Average = {avg2:.2f}%)')
# ]
# plt.legend(handles=legend_elements, loc='upper left')

# plt.savefig('../results/precision_5_boxplot.pdf')
plt.show()
plt.close()

**8. Draw bot plot for precision@10**

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('default')
data   = [precision_baseline_bm25["@10"], precision_baseline_librarian["@10"], precision_sbs["@10"]]
labels = ['Keyword-based IR', 'Librarian Agent', 'SBS-AARS']

# 顯示完整統計資訊
for i, d in enumerate(data, 1):
    stats = {
        'median': np.median(d).item(),
        'mean': np.mean(d).item(),
        'q1': np.percentile(d, 25).item(),
        'q3': np.percentile(d, 75).item()
    }

    print(stats)

plt.figure(figsize=(6, 4), dpi=300)
box = plt.boxplot(
    data,
    patch_artist=True,
    showfliers=False,
    boxprops=dict(color='black'),
    medianprops=dict(color='black', linewidth=1.5),
    whiskerprops=dict(color='black'),
    capprops=dict(color='black')
)
hatches = ['...', '///', '']
for patch, hatch in zip(box['boxes'], hatches):
    patch.set_facecolor('white')
    patch.set_hatch(hatch)
    patch.set_edgecolor('black')
plt.xticks([1, 2, 3], labels)
plt.ylabel('Precision')
plt.grid(True, axis='y', linestyle='--', alpha=0.7)
# plt.title('Baseline v.s. SBS-AARS')
# legend_elements = [
#     Patch(facecolor='white', edgecolor='black', hatch=hatches[0],
#           label=f'Baseline (Average = {avg1:.2f}%)'),
#     Patch(facecolor='white', edgecolor='black', hatch=hatches[1],
#           label=f'Smart Book Seeker (Average = {avg2:.2f}%)')
# ]
# plt.legend(handles=legend_elements, loc='upper left')

# plt.savefig('../results/precision_10_boxplot.pdf')
plt.show()
plt.close()

**9. Draw bot plot for precision@15**

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('default')
data   = [precision_baseline_bm25["@15"], precision_baseline_librarian["@15"], precision_sbs["@15"]]
labels = ['Keyword-based IR', 'Librarian Agent', 'SBS-AARS']

# 顯示完整統計資訊
for i, d in enumerate(data, 1):
    stats = {
        'median': np.median(d).item(),
        'mean': np.mean(d).item(),
        'q1': np.percentile(d, 25).item(),
        'q3': np.percentile(d, 75).item()
    }

    print(stats)

plt.figure(figsize=(6, 4), dpi=300)
box = plt.boxplot(
    data,
    patch_artist=True,
    showfliers=False,
    boxprops=dict(color='black'),
    medianprops=dict(color='black', linewidth=1.5),
    whiskerprops=dict(color='black'),
    capprops=dict(color='black')
)
hatches = ['...', '///', '']
for patch, hatch in zip(box['boxes'], hatches):
    patch.set_facecolor('white')
    patch.set_hatch(hatch)
    patch.set_edgecolor('black')
plt.xticks([1, 2, 3], labels)
plt.ylabel('Precision')
plt.ylim(0, 1)
plt.grid(True, axis='y', linestyle='--', alpha=0.7)
# plt.title('Baseline v.s. SBS-AARS')
# legend_elements = [
#     Patch(facecolor='white', edgecolor='black', hatch=hatches[0],
#           label=f'Baseline (Average = {avg1:.2f}%)'),
#     Patch(facecolor='white', edgecolor='black', hatch=hatches[1],
#           label=f'Smart Book Seeker (Average = {avg2:.2f}%)')
# ]
# plt.legend(handles=legend_elements, loc='upper left')

# plt.savefig('../results/precision_15_boxplot.pdf')
plt.show()
plt.close()

**10. Calculate recall**

In [ ]:
recall_baseline_bm25 = {}
recall_baseline_librarian = {}
recall_sbs = {}

# init
for target in targets:
    recall_baseline_bm25[target] = []
    recall_baseline_librarian[target] = []
    recall_sbs[target] = []

for target in targets:
    for i in range(100):
        temp_bm25 = 0
        temp_librarian = 0
        temp_sbs = 0

        for ans_book in result_ground_truth[target][i]["books"]:
            for book in result_baseline_bm25[target][i]["books"]:
                if book["id"] == ans_book["id"]:
                    temp_bm25 += 1 / 10
                    break
            for book in result_baseline_librarian[target][i]["books"]:
                if book["id"] == ans_book["id"]:
                    temp_librarian += 1 / 10
                    break
            for book in result_sbs[target][i]["books"]:
                if book["id"] == ans_book["id"]:
                    temp_sbs += 1 / 10
                    break

        recall_baseline_bm25[target].append(temp_bm25)
        recall_baseline_librarian[target].append(temp_librarian)
        recall_sbs[target].append(temp_sbs)

avg_recall_baseline_bm25 = {}
avg_recall_baseline_librarian = {}
avg_recall_sbs = {}

# init
for target in targets:
    avg_recall_baseline_bm25[target] = 0
    avg_recall_baseline_librarian[target] = 0
    avg_recall_sbs[target] = 0

for target in targets:
    for i in range(100):
        avg_recall_baseline_bm25[target] += recall_baseline_bm25[target][i] / 100
        avg_recall_baseline_librarian[target] += recall_baseline_librarian[target][i] / 100
        avg_recall_sbs[target] += recall_sbs[target][i] / 100

print(avg_recall_baseline_bm25)
print(avg_recall_baseline_librarian)
print(avg_recall_sbs)

**11. Calculate hit rate**

In [ ]:
hitrate_baseline_bm25 = {}
hitrate_baseline_librarian = {}
hitrate_sbs = {}

# init
for target in targets:
    hitrate_baseline_bm25[target] = []
    hitrate_baseline_librarian[target] = []
    hitrate_sbs[target] = []

for target in targets:
    for i in range(100):
        hit = False
        for book in result_baseline_bm25[target][i]["books"]:
            for ans_book in result_ground_truth[target][i]["books"]:
                if book["id"] == ans_book["id"]:
                    hit = True
                    break
            if hit:
                break
        hitrate_baseline_bm25[target].append(1 if hit else 0)

        hit = False
        for book in result_baseline_librarian[target][i]["books"]:
            for ans_book in result_ground_truth[target][i]["books"]:
                if book["id"] == ans_book["id"]:
                    hit = True
                    break
            if hit:
                break
        hitrate_baseline_librarian[target].append(1 if hit else 0)

        hit = False
        for book in result_sbs[target][i]["books"]:
            for ans_book in result_ground_truth[target][i]["books"]:
                if book["id"] == ans_book["id"]:
                    hit = True
                    break
            if hit:
                break
        hitrate_sbs[target].append(1 if hit else 0)

avg_hitrate_baseline_bm25 = {}
avg_hitrate_baseline_librarian = {}
avg_hitrate_sbs = {}

# init
for target in targets:
    avg_hitrate_baseline_bm25[target] = 0
    avg_hitrate_baseline_librarian[target] = 0
    avg_hitrate_sbs[target] = 0

for target in targets:
    for i in range(100):
        avg_hitrate_baseline_bm25[target] += hitrate_baseline_bm25[target][i] / 100
        avg_hitrate_baseline_librarian[target] += hitrate_baseline_librarian[target][i] / 100
        avg_hitrate_sbs[target] += hitrate_sbs[target][i] / 100

print(avg_hitrate_baseline_bm25)
print(avg_hitrate_baseline_librarian)
print(avg_hitrate_sbs)

**12. Calculate jaccard index**

In [ ]:
jaccard_baseline_bm25 = {}
jaccard_baseline_librarian = {}
jaccard_sbs = {}

# init
for target in targets:
    jaccard_baseline_bm25[target] = []
    jaccard_baseline_librarian[target] = []
    jaccard_sbs[target] = []

for target in targets:
    for i in range(100):
        jaccard_baseline_bm25[target].append(0)
        jaccard_baseline_librarian[target].append(0)
        jaccard_sbs[target].append(0)
        for ans_book in result_ground_truth[target][i]["books"]:
            for book in result_baseline_bm25[target][i]["books"]:
                if book["id"] == ans_book["id"]:
                    jaccard_baseline_bm25[target][i] += 1
                    break
            for book in result_baseline_librarian[target][i]["books"]:
                if book["id"] == ans_book["id"]:
                    jaccard_baseline_librarian[target][i] += 1
                    break
            for book in result_sbs[target][i]["books"]:
                if book["id"] == ans_book["id"]:
                    jaccard_sbs[target][i] += 1
                    break

        jaccard_baseline_bm25[target][i] = jaccard_baseline_bm25[target][i] / (int(target[1:]) + 10 - jaccard_baseline_bm25[target][i])
        jaccard_baseline_librarian[target][i] = jaccard_baseline_librarian[target][i] / (int(target[1:]) + 10 - jaccard_baseline_librarian[target][i])
        jaccard_sbs[target][i] = jaccard_sbs[target][i] / (int(target[1:]) + 10 - jaccard_sbs[target][i])

jaccard_baseline_bm25_avg = {}
jaccard_baseline_librarian_avg = {}
jaccard_sbs_avg = {}

# init
for target in targets:
    jaccard_baseline_bm25_avg[target] = 0
    jaccard_baseline_librarian_avg[target] = 0
    jaccard_sbs_avg[target] = 0

for target in targets:
    for i in range(100):
        jaccard_baseline_bm25_avg[target] += jaccard_baseline_bm25[target][i] / 100
        jaccard_baseline_librarian_avg[target] += jaccard_baseline_librarian[target][i] / 100
        jaccard_sbs_avg[target] += jaccard_sbs[target][i] / 100

print(jaccard_baseline_bm25_avg)
print(jaccard_baseline_librarian_avg)
print(jaccard_sbs_avg)

**13. Calculate completion rate**

In [ ]:
complete_baseline_bm25 = {}
complete_baseline_librarian = {}
complete_sbs = {}

# init
for target in targets:
    complete_baseline_bm25[target] = 0
    complete_baseline_librarian[target] = 0
    complete_sbs[target] = 0

for target in targets:
    for i in range(100):
        complete_baseline_bm25[target] += len(result_baseline_bm25[target][i]["books"]) / int(target[1:]) / 100
        complete_baseline_librarian[target] += len(result_baseline_librarian[target][i]["books"]) / int(target[1:]) / 100
        complete_sbs[target] += len(result_sbs[target][i]["books"]) / int(target[1:]) / 100

print(complete_baseline_bm25)
print(complete_baseline_librarian)
print(complete_sbs)

**14. Calculate execution time**

In [ ]:
time_baseline_librarian = {}
time_sbs = {}

# init
for target in targets:
    time_baseline_librarian[target] = 0
    time_sbs[target] = 0

for target in targets:
    for i in range(100):
        time_baseline_librarian[target] += result_baseline_librarian[target][i]["elapsed_time"] / 100
        time_sbs[target] += result_sbs[target][i]["elapsed_time"] / 100

print(time_baseline_librarian)
print(time_sbs)

**15. Calculate tokens**

In [ ]:
input_tokens_baseline_librarian = {}
output_tokens_baseline_librarian = {}
total_tokens_baseline_librarian = {}
input_tokens_sbs = {}
output_tokens_sbs = {}
total_tokens_sbs = {}

# init
for target in targets:
    input_tokens_baseline_librarian[target] = 0
    output_tokens_baseline_librarian[target] = 0
    total_tokens_baseline_librarian[target] = 0
    input_tokens_sbs[target] = 0
    output_tokens_sbs[target] = 0
    total_tokens_sbs[target] = 0

for target in targets:
    for i in range(100):
        input_tokens_baseline_librarian[target] += result_baseline_librarian[target][i]["input_tokens"] / 100
        output_tokens_baseline_librarian[target] += result_baseline_librarian[target][i]["output_tokens"] / 100
        total_tokens_baseline_librarian[target] += result_baseline_librarian[target][i]["total_tokens"] / 100
        input_tokens_sbs[target] += result_sbs[target][i]["total_input_tokens"] / 100
        output_tokens_sbs[target] += result_sbs[target][i]["total_output_tokens"] / 100
        total_tokens_sbs[target] += result_sbs[target][i]["total_tokens"] / 100

print(input_tokens_baseline_librarian)
print(output_tokens_baseline_librarian)
print(total_tokens_baseline_librarian)
print("="*20)
print(input_tokens_sbs)
print(output_tokens_sbs)
print(total_tokens_sbs)

**16. Calculate cost**

In [ ]:
cost_baseline_librarian = {}
cost_sbs = {}

# init
for target in targets:
    cost_baseline_librarian[target] = 0
    cost_sbs[target] = 0

for target in targets:
    cost_baseline_librarian[target] = (input_tokens_baseline_librarian[target] * 0.4 + output_tokens_baseline_librarian[target] * 1.6) / 1000000
    cost_sbs[target] = (input_tokens_sbs[target] * 0.4 + output_tokens_sbs[target] * 1.6) / 1000000

print(cost_baseline_librarian)
print(cost_sbs)